# IMDB Sentiment (Local): MLflow Tracking + Train

Simple baseline: **TF-IDF + Logistic Regression** trained on the IMDB dataset locally.

This notebook uses **local MLflow** for tracking params, metrics, and the model artifact.


## Start MLflow Server (Run This Before Training)

Run one of these in a terminal (not in the notebook). The project-local option avoids schema conflicts and keeps artifacts together.

1. Recommended (project-local DB + artifacts):
```bash
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlruns/mlflow.db --default-artifact-root ./mlruns
```

2. If you already started with the default `sqlite:///mlflow.db` and you get a schema error, fix it with:
```bash
mlflow db upgrade sqlite:///mlflow.db
```
(or delete `mlflow.db` if you don't need the old runs).

Open the UI:
- http://127.0.0.1:5000

This notebook logs to that server using `MLFLOW_TRACKING_URI=http://127.0.0.1:5000`.


In [2]:
# Install requirements (run once)
# %pip install -U pandas scikit-learn "mlflow<3"


In [3]:
import os
from pathlib import Path

import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


c:\Users\kvsrd\OneDrive\Desktop\Operationalize Machine Learning Models\Machine learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configure MLflow


In [4]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("imdb-sentiment-local")

print("MLflow tracking URI:", mlflow.get_tracking_uri())


MLflow tracking URI: http://127.0.0.1:5000


## Load Local Dataset

Expected file in the same folder as this notebook:
- `IMDB Dataset.csv`


In [5]:
data_path = Path("IMDB Dataset.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Missing file: {data_path.resolve()}")

df = pd.read_csv(data_path)
df = df[["review", "sentiment"]].dropna()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()

df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## Train + Log With MLflow


In [6]:
X = df["review"].astype(str)
y = df["sentiment"].map({"negative": 0, "positive": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf_max_features = 20000
tfidf_ngram_range = (1, 2)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=tfidf_max_features, ngram_range=tfidf_ngram_range)),
    ("clf", LogisticRegression(max_iter=1000)),
])

with mlflow.start_run(run_name="tfidf_logreg") as run:
    mlflow.log_param("tfidf_max_features", tfidf_max_features)
    mlflow.log_param("tfidf_ngram_range", str(tfidf_ngram_range))
    mlflow.autolog()

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    acc = accuracy_score(y_test, preds)
    mlflow.log_metric("test_accuracy", float(acc))

    input_example = pd.DataFrame({"review": ["This movie was great"]})
    mlflow.sklearn.log_model(pipeline, artifact_path="model", input_example=input_example)

    run_id = run.info.run_id

print("Run ID:", run_id)
print(f"Accuracy: {acc:.4f}")
print(classification_report(y_test, preds, target_names=["negative", "positive"]))


2026/10/24 13:43:29 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/10/24 13:43:29 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.Series'>. Dataset logging skipped.
2026/10/24 13:43:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/24 13:44:11 WARNING mlflow.sklearn: Unrecognized dataset type <class 'pandas.Series'>. Dataset logging skipped.
c:\Users\kvsrd\OneDrive\Desktop\Operationalize Machine Learning Models\Machine learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` inst

🏃 View run tfidf_logreg at: http://127.0.0.1:5000/#/experiments/1/runs/2f71773096f2497d95eac52c4fecdda5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 2f71773096f2497d95eac52c4fecdda5
Accuracy: 0.8999
              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      5000
    positive       0.89      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



## Quick Sanity Check


In [7]:
examples = [
    "This movie was fantastic. Great acting and a compelling story.",
    "Terrible film. I wasted two hours and the plot was nonsense.",
    "It was okay, not amazing but not bad either.",
]

proba = pipeline.predict_proba(examples)[:, 1]
for text, p in zip(examples, proba):
    print(f"P(positive)={p:.3f} | {text}")

    # if probability is more than 0.5 its positive sentiment else negative sentiment
    if p > 0.5:
        print("Sentiment: Positive")
    else:
        print("Sentiment: Negative")


2026/10/24 13:45:55 WARNING mlflow.sklearn: Unrecognized dataset type <class 'list'>. Dataset logging skipped.


P(positive)=0.971 | This movie was fantastic. Great acting and a compelling story.
Sentiment: Positive
P(positive)=0.009 | Terrible film. I wasted two hours and the plot was nonsense.
Sentiment: Negative
P(positive)=0.221 | It was okay, not amazing but not bad either.
Sentiment: Negative
